In [ ]:
import numpy as np  # bibliothèque pour calculs numériques (tableaux, maths)

class DecisionTreeScratch:
    def __init__(self, max_depth=2):
        # profondeur maximale de l'arbre (évite overfitting)
        self.max_depth = max_depth

        # variable qui va stocker l’arbre construit
        self.tree = None

    # =========================
    # Fonction Gini
    # =========================
    def gini(self, y):
        # récupérer les classes uniques dans y
        classes = np.unique(y)

        # valeur initiale du gini
        gini = 1.0

        # calcul du Gini impurity
        for c in classes:
            p = np.sum(y == c) / len(y)  # probabilité de la classe c
            gini -= p ** 2               # formule de Gini

        return gini


    def best_split(self, X, y):
        best_feature = None      # meilleure variable (colonne)
        best_threshold = None    # meilleur seuil
        best_gini = 999          # valeur très grande au départ

        n_samples, n_features = X.shape  # taille des données

        # parcourir toutes les features
        for feature in range(n_features):

            # tester tous les seuils possibles
            thresholds = np.unique(X[:, feature])

            for t in thresholds:

                # séparation gauche / droite
                left_mask = X[:, feature] <= t
                right_mask = X[:, feature] > t

                # éviter les splits vides
                if len(y[left_mask]) == 0 or len(y[right_mask]) == 0:
                    continue

                # calcul Gini pour gauche et droite
                gini_left = self.gini(y[left_mask])
                gini_right = self.gini(y[right_mask])

                # Gini total pondéré
                gini_total = (
                    len(y[left_mask]) * gini_left +
                    len(y[right_mask]) * gini_right
                ) / len(y)

                # garder le meilleur split (le plus pur)
                if gini_total < best_gini:
                    best_gini = gini_total
                    best_feature = feature
                    best_threshold = t

        return best_feature, best_threshold

 
    def build_tree(self, X, y, depth=0):

        # condition d'arrêt :
        # - profondeur max atteinte
        # - ou toutes les classes identiques
        if depth >= self.max_depth or len(np.unique(y)) == 1:
            return np.bincount(y).argmax()  # classe majoritaire

        # trouver meilleur split
        feature, threshold = self.best_split(X, y)

        # si aucun split possible
        if feature is None:
            return np.bincount(y).argmax()

        # séparation des données
        left_mask = X[:, feature] <= threshold
        right_mask = X[:, feature] > threshold

        # construire sous-arbre gauche
        left_tree = self.build_tree(X[left_mask], y[left_mask], depth + 1)

        # construire sous-arbre droit
        right_tree = self.build_tree(X[right_mask], y[right_mask], depth + 1)

        # retourner le nœud courant
        return {
            "feature": feature,        # variable utilisée
            "threshold": threshold,    # seuil de séparation
            "left": left_tree,         # branche gauche
            "right": right_tree        # branche droite
        }

    def fit(self, X, y):
        # construire l'arbre complet
        self.tree = self.build_tree(X, y)

   
    def predict_one(self, x, tree):

        # si on arrive à une feuille (classe finale)
        if not isinstance(tree, dict):
            return tree

        # récupérer feature et seuil
        feature = tree["feature"]
        threshold = tree["threshold"]

        # aller à gauche ou droite selon la condition
        if x[feature] <= threshold:
            return self.predict_one(x, tree["left"])
        else:
            return self.predict_one(x, tree["right"])


    def predict(self, X):
        return np.array([self.predict_one(x, self.tree) for x in X])

In [ ]:
import numpy as np

# =========================
#  Données
# =========================
X = np.array([
    [20, 1],
    [25, 1],
    [30, 0],
    [35, 0],
    [40, 1],
    [50, 0]
])

y = np.array([0, 0, 0, 1, 1, 1])

# =========================
#  Modèle
# =========================
model = DecisionTreeScratch(max_depth=2)

# =========================
# Entraînement
# =========================
model.fit(X, y)

# =========================
#  Prédiction
# =========================
X_test = np.array([[22, 1], [45, 0]])

print("Prédictions :", model.predict(X_test))